# Longor VLAbench launcher (`select_fruit`)

Runs the full Longor path (planner + executor + VLA service) on the default VLAbench task.

Launcher contract: `RUN_SCRIPT=examples/run_vlabench_task.py`

## 1 — Clone repo (Colab secrets)

Add secrets in Colab (**Secrets** in the left sidebar): `GITHUB_TOKEN` (repo-scope PAT) and `OPENAI_API_KEY`.

In [ ]:
import os
from google.colab import userdata, drive

github_token = userdata.get('GITHUB_TOKEN')
openai_api_key = userdata.get('OPENAI_API_KEY')

if os.path.exists('/content/Longor'):
    os.chdir('/content/Longor')
    print('pulling repo...')
    !git pull
else:
    os.environ['CLONE_URL'] = f"https://{github_token}@github.com/pmish11/Longor-Robotics.git"
    !git clone $CLONE_URL "/content/Longor"
    os.chdir('/content/Longor')

# for bypassing FileURLRetrievalError
drive.mount('/content/drive')

## 2 — Headless rendering + app deps (VLAbench)

In [ ]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

In [ ]:
%%bash
set -euo pipefail

cd /content/Longor
APP_VENV=/content/Longor/.venv
VLABENCH_PIN=cf588fe60c0c7282174fe979f5913170cfe69017
RRT_PIN=e51d95ee489a225220d6ae2a764c4111f6ba7d85

apt-get update
apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev python3.10-venv

# If .venv is broken: rm -rf .venv then re-run this cell.
if [ ! -d .venv ]; then
  python3.10 -m venv .venv || {
    python3.10 -m venv --without-pip .venv
    curl -fsSL https://bootstrap.pypa.io/get-pip.py | .venv/bin/python
  }
fi

source .venv/bin/activate

if [ ! -d .vendor/VLABench/.git ]; then
  git clone https://github.com/OpenMOSS/VLABench .vendor/VLABench
fi
git -C .vendor/VLABench checkout "$VLABENCH_PIN"

if [ ! -d .vendor/rrt-algorithms/.git ]; then
  git clone https://github.com/motion-planning/rrt-algorithms .vendor/rrt-algorithms
fi
git -C .vendor/rrt-algorithms checkout $RRT_PIN

python -m pip install -U pip
python -m pip install -r deps/requirements-vlabench.txt
touch .vendor/rrt-algorithms/rrt_algorithms/rrt/__init__.py
touch .vendor/rrt-algorithms/rrt_algorithms/search_space/__init__.py
touch .vendor/rrt-algorithms/rrt_algorithms/utilities/__init__.py
python -m pip install -e .vendor/rrt-algorithms
python -m pip install -e .vendor/VLABench
python -m pip install -e .
# bypassing FileURLRetrievalError
ASSETS_DIR=".vendor/VLABench/VLABench/assets"
ASSETS_TAR="/content/drive/MyDrive/Colab/assets.tar"

if find "$ASSETS_DIR/obj/meshes/fruit" -name '*.xml' -print -quit | grep -q .; then
  echo "assets already present, skipping extract"
else
  test -f "$ASSETS_TAR" || { echo "missing $ASSETS_TAR — mount Drive first" >&2; exit 1; }
  mkdir -p ".vendor/VLABench/VLABench"
  tar -xf "$ASSETS_TAR" -C ".vendor/VLABench/VLABench"
fi

python -c "import VLABench.robots, VLABench.tasks; from VLABench.envs import load_env; print('section 2 OK: VLABench importable')"

## 3 — SmolVLA policy venv

In [ ]:
%%bash
set -euo pipefail

cd /content/Longor
VLA_VENV=/content/Longor/.vla-venv
VLA_PY="$VLA_VENV/bin/python"

# If .vla-venv is broken: rm -rf "$VLA_VENV" then re-run this cell.
# ensurepip stderr on Colab is expected; the fallback below installs pip.
if [ ! -d "$VLA_VENV" ]; then
  python3 -m venv "$VLA_VENV" || {
    python3 -m venv --without-pip "$VLA_VENV"
    curl -fsSL https://bootstrap.pypa.io/get-pip.py | "$VLA_PY"
  }
fi

"$VLA_PY" -m pip install -U pip
"$VLA_PY" -m pip install -r deps/requirements-vla-smolvla-linux.txt
"$VLA_PY" -m pip install -e .

## 4 — Configure `.env` for Longor run

In [ ]:
from pathlib import Path

Path('.env').write_text(
    f'OPENAI_API_KEY={openai_api_key}\n'
    'LOG_LEVEL=DEBUG\n'
    'LLM_MODEL=gpt-4.1-mini\n'
    'SAVE_VIDEO=1\n',
    encoding='utf-8',
)
print('.env written')

## 5 — Start VLA server + health

In [ ]:
%%bash
set -euo pipefail

cd /content/Longor
VLA_LOG=/tmp/longor_vlabench_vla_server.log
VLA_PYTHON=/content/Longor/.vla-venv/bin/python

if curl -fsS http://localhost:8765/health; then
  echo
  echo "VLA server already healthy"
  exit 0
fi

pkill -f "services/run_vla_server.py" || true
for _ in $(seq 1 60); do
  pgrep -f "services/run_vla_server.py" >/dev/null || break
  sleep 1
done
if pgrep -f "services/run_vla_server.py" >/dev/null; then
  echo "leftover VLA server still running; not starting another" >&2
  exit 1
fi

if ! "$VLA_PYTHON" -c "import config" 2>/dev/null; then
  echo "VLA venv missing longor install — re-run section 3 (or rm -rf .vla-venv first)" >&2
  exit 1
fi

nohup "$VLA_PYTHON" services/run_vla_server.py >"$VLA_LOG" 2>&1 &

sleep 2
if ! pgrep -f "services/run_vla_server.py" >/dev/null; then
  echo "VLA server exited immediately:" >&2
  tail -50 "$VLA_LOG" >&2
  exit 1
fi

elapsed=0
for _ in $(seq 1 180); do
  if curl -fsS http://localhost:8765/health; then
    echo
    echo "VLA server healthy"
    exit 0
  fi
  if ! pgrep -f "services/run_vla_server.py" >/dev/null; then
    echo "VLA server exited during startup:" >&2
    tail -50 "$VLA_LOG" >&2
    exit 1
  fi
  elapsed=$((elapsed + 5))
  echo "waiting for server to start... (${elapsed}s)"
  sleep 5
done

echo "VLA server not healthy after ${elapsed}s; see $VLA_LOG" >&2
tail -50 "$VLA_LOG" >&2
exit 1

## 6 — Run Longor once

In [ ]:
!/content/Longor/.venv/bin/python examples/run_vlabench_task.py --task "select_fruit"

## 7 — Stop VLA server

In [ ]:
!killall -vw python

## 8 — M4 parity artifact (Longor subtask path)

After section 6, record one rollout artifact for the parity gate. Set `RUN_SUCCESS=1` if the run succeeded, else `0`.


In [ ]:
import json
from pathlib import Path

RUN_SUCCESS = 0  # set to 1 after a successful select_fruit rollout
ARTIFACT_DIR = Path('artifacts/m4_parity')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
log_candidates = sorted(Path('.').glob('run*.log'), key=lambda p: p.stat().st_mtime, reverse=True)
log_path = log_candidates[0] if log_candidates else Path('run.log')

payload = {
    'gate': 'm4_parity',
    'track': 'longor_subtask',
    'task': 'select_fruit',
    'policy': 'lerobot/smolvla_vlabench',
    'vlabench_sha': 'cf588fe60c0c7282174fe979f5913170cfe69017',
    'prompt_path': 'planner_subtask_pick_up_fruit',
    'n_episodes': 1,
    'successes': int(RUN_SUCCESS),
    'success_rate': float(RUN_SUCCESS),
    'log_path': str(log_path),
    'video_path': 'run.mp4',
}
out = ARTIFACT_DIR / 'longor_subtask.json'
out.write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')
print(json.dumps(payload, indent=2))
print('artifact:', out)

## 9 — Save the video

### HOW TO GET VIDEO FILES (when running via Colab extension)?
1. run with 'SAVE_VIDEO=1'
2. click Colab -> mount server to workspace
3. reload IDE window.
4. open the files sidebar! :)